In [1]:
print(1)

1


In [ ]:
import pandas as pd
import numpy as np
import os

os.makedirs("data", exist_ok=True)

# -----------------------------
# COMMON SETTINGS
# -----------------------------
N = 500  # number of rows

timestamps = pd.date_range(start="2026-01-01", periods=N, freq="h")

# -----------------------------
# ☀️ SOLAR DATASET
# -----------------------------
solar_rows = []
prev_output = 0

for t in timestamps:
    hour = t.hour

    temp = np.random.uniform(25, 40)
    humidity = np.random.uniform(40, 90)
    clouds = np.random.uniform(0, 100)
    wind = np.random.uniform(0, 10)

    is_day = 1 if 6 <= hour <= 18 else 0

    if is_day:
        solar_output = max(0, (100 - clouds)/100 * (hour/12) * 5 + np.random.normal(0, 0.2))
    else:
        solar_output = 0

    solar_rows.append([
        t, temp, humidity, clouds, wind,
        hour, is_day, prev_output, round(solar_output, 2)
    ])

    prev_output = solar_output

solar_df = pd.DataFrame(solar_rows, columns=[
    "timestamp","temperature","humidity","clouds",
    "wind_speed","hour","is_day",
    "historical_output","solar_output"
])

solar_df.to_csv("data/solar.csv", index=False)


# -----------------------------
# ⚡ LOAD DATASET (MOST IMPORTANT)
# -----------------------------
load_rows = []
prev_load = 2

for t in timestamps:
    hour = t.hour
    day = t.dayofweek

    temp = np.random.uniform(25, 40)
    humidity = np.random.uniform(40, 90)

    # realistic load pattern
    base = 2 + 2*np.sin((hour/24)*2*np.pi)
    noise = np.random.normal(0, 0.3)

    load = max(0.5, base + noise + (0.5 if day >= 5 else 0))

    load_rows.append([
        t, hour, day, temp, humidity,
        prev_load, round(load, 2)
    ])

    prev_load = load

load_df = pd.DataFrame(load_rows, columns=[
    "timestamp","hour","day",
    "temperature","humidity",
    "historical_load","load"
])

load_df.to_csv("data/load.csv", index=False)


# -----------------------------
# 🌬 WIND DATASET
# -----------------------------
wind_rows = []

for t in timestamps:
    temp = np.random.uniform(20, 35)
    pressure = np.random.uniform(1000, 1020)
    wind_speed = np.random.uniform(0, 15)

    wind_output = wind_speed**3 / 100 + np.random.normal(0, 0.5)

    wind_rows.append([t, temp, pressure, wind_speed, round(wind_output, 2)])

wind_df = pd.DataFrame(wind_rows, columns=[
    "timestamp","temperature","pressure",
    "wind_speed","wind_output"
])

wind_df.to_csv("data/wind.csv", index=False)


# -----------------------------
# 🚨 ANOMALY DATASET (THEFT)
# -----------------------------
anomaly_rows = []

for t in timestamps:
    voltage = np.random.uniform(210, 240)
    current = np.random.uniform(1, 10)
    consumption = voltage * current / 1000

    # inject anomalies
    if np.random.rand() < 0.05:
        consumption *= 3  # abnormal spike

    anomaly_rows.append([t, voltage, current, round(consumption, 2)])

anomaly_df = pd.DataFrame(anomaly_rows, columns=[
    "timestamp","voltage","current","consumption"
])

anomaly_df.to_csv("data/anomaly.csv", index=False)


# -----------------------------
# ⚠️ POWER CUT DATASET
# -----------------------------
powercut_rows = []

for t in timestamps:
    voltage = np.random.uniform(180, 240)
    wind = np.random.uniform(0, 20)
    rain = np.random.uniform(0, 50)

    # rule-based label
    power_cut = 1 if voltage < 200 or wind > 15 or rain > 40 else 0

    powercut_rows.append([t, voltage, wind, rain, power_cut])

powercut_df = pd.DataFrame(powercut_rows, columns=[
    "timestamp","voltage","wind_speed","rainfall","power_cut"
])

powercut_df.to_csv("data/powercut.csv", index=False)


print("🔥 ALL DATASETS GENERATED SUCCESSFULLY")

🔥 ALL DATASETS GENERATED SUCCESSFULLY


In [14]:
import requests
import pandas as pd
import numpy as np
import os

os.makedirs("data", exist_ok=True)

url = "https://power.larc.nasa.gov/api/temporal/daily/point"

params = {
    "parameters": "ALLSKY_SFC_SW_DWN,T2M,WS2M",
    "community": "RE",
    "latitude": 11.92237,
    "longitude": 79.60673,
    "start": 20250101,
    "end": 20251231,
    "format": "JSON"
}

response = requests.get(url, params=params)
data = response.json()

irradiance = data["properties"]["parameter"]["ALLSKY_SFC_SW_DWN"]
temp = data["properties"]["parameter"]["T2M"]
wind = data["properties"]["parameter"]["WS2M"]

rows = []
prev_efficiency = 0

# 🔥 global max for scaling
max_irrad = max(irradiance.values())

for date in irradiance.keys():

    daily_irrad = irradiance[date]
    temperature = temp[date]
    wind_speed = wind[date]

    for hour in range(24):

        # ❌ skip night (important)
        if not (6 <= hour <= 18):
            continue

        # solar curve
        factor = np.sin((hour - 6) / 12 * np.pi)

        solar_output = daily_irrad * factor

        efficiency = (solar_output / max_irrad) * 100
        efficiency = max(0, min(efficiency, 100))

        rows.append([
            date,
            hour,
            temperature,
            wind_speed,
            daily_irrad,
            prev_efficiency,
            round(efficiency, 2)
        ])

        prev_efficiency = efficiency

df = pd.DataFrame(rows, columns=[
    "date",
    "hour",
    "temperature",
    "wind_speed",
    "irradiance",
    "historical_efficiency",
    "efficiency"
])

df.to_csv("data/solar_2025.csv", index=False)

print("🔥 Dataset created with proper efficiency!")

🔥 Dataset created with proper efficiency!


In [15]:
df = pd.read_csv('data/solar_2025.csv')
df.describe()

,date,hour,temperature,wind_speed,irradiance,historical_efficiency,efficiency
count,4.745000e+03,4745.000000,4745.000000,4745.000000,4745.000000,4745.000000,4745.000000
mean,2.025067e+07,12.000000,27.662356,2.377205,5.131927,40.944472,40.944466
std,3.450382e+02,3.742052,2.836415,0.803241,1.331228,27.250984,27.250989
min,2.025010e+07,6.000000,21.370000,0.710000,0.902200,0.000000,0.000000
25%,2.025040e+07,9.000000,25.390000,1.770000,4.211000,19.035219,19.040000
50%,2.025070e+07,12.000000,28.130000,2.220000,5.365200,41.733348,41.730000
75%,2.025100e+07,15.000000,30.010000,2.820000,6.177400,63.401507,63.400000
max,2.025123e+07,18.000000,32.460000,6.080000,7.323400,100.000000,100.000000


In [1]:
import requests
import pandas as pd
import numpy as np
import os

os.makedirs("data", exist_ok=True)

url = "https://power.larc.nasa.gov/api/temporal/daily/point"

params = {
    "parameters": "WS2M,T2M",
    "community": "RE",
    "latitude": 11.92237,
    "longitude": 79.60673,
    "start": 20250101,
    "end": 20251231,
    "format": "JSON"
}

response = requests.get(url, params=params)
data = response.json()

wind = data["properties"]["parameter"]["WS2M"]
temp = data["properties"]["parameter"]["T2M"]

rows = []
prev_eff = 0

max_wind = max(wind.values())  # for scaling

for date in wind.keys():

    wind_speed = wind[date]
    temperature = temp[date]

    # simulate hourly variation
    for hour in range(24):

        # wind fluctuation
        fluctuation = np.random.uniform(0.7, 1.3)
        wind_hour = wind_speed * fluctuation

        # ⚡ wind power ~ cube relation
        power = wind_hour ** 3

        efficiency = (power / (max_wind ** 3)) * 100
        efficiency = max(0, min(efficiency, 100))

        rows.append([
            date,
            hour,
            temperature,
            wind_hour,
            prev_eff,
            round(efficiency, 2)
        ])

        prev_eff = efficiency

df = pd.DataFrame(rows, columns=[
    "date",
    "hour",
    "temperature",
    "wind_speed",
    "historical_efficiency",
    "efficiency"
])

df.to_csv("data/wind_2025.csv", index=False)

print("🌬️ Wind dataset created!")

🌬️ Wind dataset created!
